In [ ]:

# ============================================================
# USER STORY 1: PHÂN TÍCH VẬN HÀNH LƯỚI ĐIỆN
# Operational Manager
# ============================================================

# 1. KHUNG THIẾT KẾ WHAT - WHY - HOW
#
# WHAT - Dữ liệu:
# - Chuỗi thời gian đa cấp: giờ, ngày, tháng, mùa.
# - 6 biến phụ tải: HUFL, HULL, MUFL, MULL, LUFL, LULL.
# - Tỷ số tải hữu công/vô công.
# - Chuỗi nhiệt độ dầu OT và ma trận tương quan trễ.
#
# WHY - Tác vụ phân tích:
# - Nhận diện xu hướng và tính mùa vụ của phụ tải.
# - So sánh tỷ trọng tổn thất vô công giữa 3 cấp điện áp.
# - Xác định độ trễ quán tính nhiệt của trạm biến áp.
#
# HOW - Phương pháp trực quan hóa:
# - Hourly Profile + Historical Quantile Ribbon.
# - 100% Stacked Bar/Area Chart.
# - Lag Cross-Correlation Heatmap.


# ============================================================
# 2. KHAI BÁO THƯ VIỆN
# ============================================================

import os
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# ============================================================
# 3. THIẾT LẬP ĐƯỜNG DẪN VÀ NẠP DỮ LIỆU
# ============================================================

BASE_DIR = os.path.abspath("..")

ARTIFACTS_DIR = os.path.join(
    BASE_DIR,
    "output",
    "artifacts",
    "ETTh1"
)

clean_df = pd.read_parquet(
    os.path.join(ARTIFACTS_DIR, "clean_ett.parquet")
)

quantiles_df = pd.read_parquet(
    os.path.join(ARTIFACTS_DIR, "quantile_bands.parquet")
)

with open(
    os.path.join(
        ARTIFACTS_DIR,
        "lag_correlation_matrix.json"
    ),
    "r",
    encoding="utf-8"
) as f:
    lag_df = pd.DataFrame(json.load(f))

print(
    f"Nạp thành công dataset ETTh1: "
    f"{clean_df.shape[0]} bản ghi."
)


# ============================================================
# 4. KHÁM PHÁ 1: HISTORICAL QUANTILE RIBBON (FAN CHART)
# ============================================================

q_summer = quantiles_df[
    quantiles_df["season"] == "Summer"
]

fig_fan = go.Figure()

# Dải phân vị 5% - 95% (90% lịch sử)
fig_fan.add_trace(
    go.Scatter(
        x=q_summer["hour"],
        y=q_summer["OT_q95"],
        mode="lines",
        line=dict(width=0),
        showlegend=False
    )
)

fig_fan.add_trace(
    go.Scatter(
        x=q_summer["hour"],
        y=q_summer["OT_q05"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(30, 58, 138, 0.14)",
        name="Dải 90% lịch sử (5%-95%)"
    )
)

# Dải phân vị 25% - 75% (IQR)
fig_fan.add_trace(
    go.Scatter(
        x=q_summer["hour"],
        y=q_summer["OT_q75"],
        mode="lines",
        line=dict(width=0),
        showlegend=False
    )
)

fig_fan.add_trace(
    go.Scatter(
        x=q_summer["hour"],
        y=q_summer["OT_q25"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(30, 58, 138, 0.28)",
        name="Dải 50% IQR (25%-75%)"
    )
)

# Đường nhiệt độ trung vị
fig_fan.add_trace(
    go.Scatter(
        x=q_summer["hour"],
        y=q_summer["OT_q50"],
        mode="lines+markers",
        line=dict(color="#1E3A8A", width=2.5),
        name="Nhiệt độ trung vị (P50)"
    )
)

# Thiết lập giao diện biểu đồ
fig_fan.update_layout(
    title=(
        "Action Title: Nhiệt độ dầu OT đạt đỉnh "
        "vào khung 14h-17h hàng ngày, với dải biến thiên "
        "90% dao động từ 18°C đến 38°C vào mùa hè"
    ),
    xaxis_title="Khung giờ trong ngày (0 - 23h)",
    yaxis_title="Nhiệt độ dầu biến áp OT (°C)",
    template="plotly_white",
    hovermode="x unified"
)

fig_fan.show()


# ============================================================
# 5. KHÁM PHÁ 2: LAG CROSS-CORRELATION HEATMAP
# ============================================================

load_cols = [
    "HUFL",
    "HULL",
    "MUFL",
    "MULL",
    "LUFL",
    "LULL"
]

corr_matrix = lag_df[load_cols].values.T

fig_lag = px.imshow(
    corr_matrix,
    x=lag_df["lag_hours"],
    y=load_cols,
    color_continuous_scale="Viridis",
    labels={
        "x": "Độ trễ thời gian (Giờ)",
        "y": "Nhánh phụ tải",
        "color": "Pearson Corr"
    },
    title=(
        "Action Title: Phân tích độ trễ quán tính nhiệt "
        "của trạm biến áp đối với các nhánh phụ tải"
    )
)

fig_lag.update_layout(
    template="plotly_white",
    xaxis=dict(dtick=1)
)

fig_lag.show()
